In [1]:
!pip install -q langchain
!pip install -q langgraph
!pip install -q langchain-ollama
!pip install -q langchain-community
!pip install -q wikipedia
!pip install -q duckduckgo-search

In [2]:
from typing import TypedDict

from langchain_ollama import ChatOllama

from langgraph.graph import StateGraph, START, END

from langchain_core.messages import HumanMessage, AIMessage

c:\Users\khana\OneDrive\Desktop\GPU t est in Tensorflow\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\khana\OneDrive\Desktop\GPU t est in Tensorflow\.venv\lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
llm = ChatOllama(
    model="deepseek-r1:1.5b",
    temperature=0
)

print("✅ DeepSeek-R1 Loaded Successfully")

✅ DeepSeek-R1 Loaded Successfully


In [4]:
class AgentState(TypedDict):
    question: str
    research: str
    code: str
    writing: str
    review: str
    final_answer: str

In [5]:
def research_agent(state: AgentState):

    prompt = f"""
You are an AI Researcher.

Research the following topic.

Question:
{state["question"]}

Give a detailed answer.
"""

    response = llm.invoke(prompt)

    state["research"] = response.content

    return state

In [6]:
def coding_agent(state: AgentState):

    prompt = f"""
You are an Expert Python Programmer.

Question:

{state["question"]}

If coding is required,
write clean Python code.

Otherwise explain.
"""

    response = llm.invoke(prompt)

    state["code"] = response.content

    return state

In [7]:
def writing_agent(state: AgentState):

    prompt = f"""
You are a Professional Writer.

Question:

{state["question"]}

Research:

{state["research"]}

Write a professional answer.
"""

    response = llm.invoke(prompt)

    state["writing"] = response.content

    return state

In [8]:
def reviewer_agent(state: AgentState):

    prompt = f"""
You are a Quality Reviewer.

Research

{state["research"]}

Writing

{state["writing"]}

Code

{state["code"]}

Create the final polished answer.
"""

    response = llm.invoke(prompt)

    state["review"] = response.content
    state["final_answer"] = response.content

    return state

In [9]:
graph = StateGraph(AgentState)

graph.add_node("Research", research_agent)
graph.add_node("Coder", coding_agent)
graph.add_node("Writer", writing_agent)
graph.add_node("Reviewer", reviewer_agent)

graph.add_edge(START, "Research")
graph.add_edge("Research", "Coder")
graph.add_edge("Coder", "Writer")
graph.add_edge("Writer", "Reviewer")
graph.add_edge("Reviewer", END)

app = graph.compile()

print("✅ Multi-Agent Graph Created Successfully")

✅ Multi-Agent Graph Created Successfully


In [10]:
question = input("Ask your question: ")

result = app.invoke({
    "question": question,
    "research": "",
    "code": "",
    "writing": "",
    "review": "",
    "final_answer": ""
})

print("\n==============================")
print("FINAL ANSWER")
print("==============================\n")

print(result["final_answer"])


FINAL ANSWER



To calculate the area of a circle given its radius in Python, you can use the formula \( \text{Area} = \pi r^2 \). Here's how to implement it:

```python
import math

radius = float(input("Enter the radius of the circle: "))
area = math.pi * radius ** 2
print(f"The area of the circle with radius {radius} is {area}")
```

**Explanation:**

1. **Import Math Module:** The `math` module provides the value of π (pi) which is approximately 3.14159.
2. **Read Input:** The user is prompted to enter the radius of the circle.
3. **Calculate Area:** Using the formula \( \text{Area} = \pi r^2 \), compute the area using `math.pi * radius ** 2`.
4. **Output Result:** Print the calculated area with a message indicating the radius used.

This code will accurately compute and display the area of a circle based on the provided radius.


In [11]:
def supervisor_agent(state: AgentState):

    question = state["question"].lower()

    if any(word in question for word in [
        "code",
        "python",
        "program",
        "function",
        "bug",
        "error",
        "java",
        "c++",
        "html",
        "css"
    ]):
        state["next"] = "Coder"

    elif any(word in question for word in [
        "write",
        "essay",
        "email",
        "article",
        "report",
        "blog"
    ]):
        state["next"] = "Writer"

    else:
        state["next"] = "Research"

    return state

In [12]:
from typing import TypedDict

class AgentState(TypedDict):
    question: str

    research: str
    code: str
    writing: str
    review: str

    final_answer: str

    next: str

In [13]:
def router(state: AgentState):

    return state["next"]

In [14]:
graph = StateGraph(AgentState)

graph.add_node("Supervisor", supervisor_agent)
graph.add_node("Research", research_agent)
graph.add_node("Coder", coding_agent)
graph.add_node("Writer", writing_agent)
graph.add_node("Reviewer", reviewer_agent)

graph.add_edge(START, "Supervisor")

graph.add_conditional_edges(
    "Supervisor",
    router,
    {
        "Research": "Research",
        "Coder": "Coder",
        "Writer": "Writer"
    }
)

graph.add_edge("Research", "Reviewer")
graph.add_edge("Coder", "Reviewer")
graph.add_edge("Writer", "Reviewer")

graph.add_edge("Reviewer", END)

app = graph.compile()

print("Graph Ready")

Graph Ready


In [19]:
question = input("Ask Anything : ")

result = app.invoke(
    {
        "question": question,
        "research": "",
        "code": "",
        "writing": "",
        "review": "",
        "final_answer": "",
        "next": ""
    }
)

print("\n")
print(result["final_answer"])





To create a Hello World program in Python, you can use the `print` function to display the string "Hello World". Here's how it looks:

```python
print("Hello World")
```

When you run this code, it will output:
```
Hello World
```
